In [1]:
# 1. IMPORTAÇÕES E CONFIGURAÇÕES INICIAIS
# =============================================================================
import os
import re
import gc
import sys
import shutil
import pickle
import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.preprocessing import StandardScaler
from src.mggp import MGGP

plt.switch_backend('Agg')

print('Importados!!!')

Importados!!!


In [ ]:
# 2. CAMINHOS E BANCO DE DADOS
# =============================================================================
DIRETORIO_BASE = r"C:\IC_alex\TreinamentoMGGP_2_0\RESULTADOS"
CAMINHO_PLANILHA = r"C:\IC_alex\TreinamentoMGGP_2_0\controle_treinamentos.xlsx"

caminho_treino = r'C:\IC_alex\Database\wang21dv_bic_MGGP.xlsx'
caminho_validacao = r'C:\IC_alex\Database\wang22dv_bic_MGGP.xlsx'

pistas_validacao_massa = {
    "Wang 2.1 (Treino)": r'C:\IC_alex\Database\wang21dv_bic_MGGP.xlsx',
    "Wang 2.2 (Validação)": r'C:\IC_alex\Database\wang22dv_bic_MGGP.xlsx',
    "Wang 3.1 (Teste A)": r'C:\IC_alex\Database\wang31dv_bic_MGGP.xlsx',
    "Wang 8.1 (Teste B)": r'C:\IC_alex\Database\wang81dv_bic_MGGP.xlsx'
}

# =============================================================================
# 3. FUNÇÕES AUXILIARES
# =============================================================================

def carregar_dados_bicicleta(caminho_arquivo):
    df = pd.read_excel(caminho_arquivo, header=None)
    u = df.iloc[:, [14, 2, 22, 23, 11]].values
    y_raw = df.iloc[:, [18, 17]].values
    return u, y_raw  # Retornando os dados puramente originais

def calcular_rmse(y_true, y_pred):
    """Calcula a Raiz do Erro Quadrático Médio"""
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def atualizar_painel(id_atual, n_terms, max_height, pendentes_restantes, tentativa, aprovados):
    sys.stdout.write("\033[s")
    sys.stdout.write("\033[H")
    print("=" * 85)
    print(f" 📑  SISTEMA AUTÔNOMO MGGP — LENDO PLANILHA EM TEMPO REAL")
    print("=" * 85)
    print(f" 📈 Restam: {pendentes_restantes} treinamentos pendentes na fila.")
    print(f" 🚀 Processando: [ ID: {id_atual} | nTerms: {n_terms} | maxHeight: {max_height} ]")
    print(f" 🎯 Fábrica: [ Aprovados: {aprovados}/10 | Tentativa Atual: {tentativa}/50 ]")
    print("=" * 85)
    print(" 📜 LOGS DA RODADA ATUAL:")
    print("-" * 85)
    sys.stdout.write("\033[u")
    sys.stdout.flush()

# =============================================================================
# 4. MOTOR PRINCIPAL
# =============================================================================
if __name__ == "__main__":
    os.system('cls' if os.name == 'nt' else 'clear')
    print("\n" * 8)
    
    # 🔁 LOOP(): INICIA AQUI
    while True:
        
        # --- 1. LÊ A PLANILHA FRESCA DO HD ---
        if not os.path.exists(CAMINHO_PLANILHA):
            print(f"❌ Erro: Planilha '{CAMINHO_PLANILHA}' sumiu!")
            break
            
        df_controle = pd.read_excel(CAMINHO_PLANILHA)
        df_pendentes = df_controle[df_controle['Status'].astype(str).str.strip().str.upper() == 'PENDENTE']
        
        if df_pendentes.empty:
            print("\n🏁 Fim da linha! Não há mais treinamentos 'Pendente' na planilha.")
            break
            
        # Pega a primeira tarefa pendente da lista
        linha_atual = df_pendentes.iloc[0]
        t_id = int(linha_atual['ID'])
        n_atual = int(linha_atual['nTerms'])
        h_atual = int(linha_atual['maxHeight'])
        pendentes_restantes = len(df_pendentes)
        
        # --- 2. MODIFICA PARÂMETROS E CRIA CHECKLIST LOCAL ---
        pasta_salvamento = os.path.join(DIRETORIO_BASE, f"Treinamento_{t_id}")
        pasta_training = os.path.join(pasta_salvamento, "TRAINING")
        os.makedirs(pasta_training, exist_ok=True)
        
        # [NOVIDADE]: GERA OU LÊ A PLANILHA LOCAL DE CHECKPOINT (AS 20 ETAPAS)
        caminho_check_local = os.path.join(pasta_salvamento, "checklist_local.xlsx")
        if not os.path.exists(caminho_check_local):
            etapas_treino = [f"Treino_{i:02d}" for i in range(1, 11)]
            etapas_validacao = [f"Validacao_{i:02d}" for i in range(1, 11)]
            df_local = pd.DataFrame({
                "Etapa": etapas_treino + etapas_validacao,
                "Status": ["Pendente"] * 20
            })
            df_local.to_excel(caminho_check_local, index=False)
        else:
            df_local = pd.read_excel(caminho_check_local)

        parametros_mggp = {
            "nDelays": [1, 2, 5, 10],
            "generations": 300, 
            "populationSize": 300,
            "evaluationMode": "RMSE", 
            "k": 300, 
            "evaluationType": "MShooting",
            "evaluationTypeTest": "FreeRun", 
            "nTerms": n_atual, 
            "maxHeight": h_atual,
            "mutationRate": 0.3, 
            "crossoverRate": 0.8,
            "elitePercentage": 10,
            "mode": "MIMO", 
            "froe_mode": False, 
            "operators": ['add', 'subtraction', 'mul']
        }
        pd.DataFrame(list(parametros_mggp.items()), columns=['Parametro', 'Valor']).to_csv(os.path.join(pasta_salvamento, "parametros_utilizados.csv"), index=False)
        
        u_train, y_train = carregar_dados_bicicleta(caminho_treino)
        u_val, y_val = carregar_dados_bicicleta(caminho_validacao)
        
        # --- 3. FÁBRICA: 10 TREINAMENTOS COM OS MESMOS PARÂMETROS ---
        meta_modelos = 10
        
        # [NOVIDADE]: Verifica quantos treinos já foram aprovados olhando a planilha local
        treinos_concluidos = df_local[df_local['Etapa'].str.startswith('Treino_') & (df_local['Status'] == 'Concluído')].shape[0]
        modelos_aprovados = treinos_concluidos
        tentativas = 0
        
        while modelos_aprovados < meta_modelos and tentativas < 50:
            tentativas += 1
            atualizar_painel(t_id, n_atual, h_atual, pendentes_restantes, tentativas, modelos_aprovados)
            
            caminho_temp = os.path.join(pasta_salvamento, "temp_mggp.pkl")
            mggp = MGGP(inputs=u_train, outputs=y_train, validation=(u_val, y_val), filename=caminho_temp, **parametros_mggp)
            mggp.run()
            
            best_model = mggp._hof[0]
            args_val = (y_val, u_val) if parametros_mggp["evaluationTypeTest"] != "MShooting" else (parametros_mggp["k"], y_val, u_val)
            
            erro_v = 999.0
            try:
                yp, yd = best_model.predict(parametros_mggp["evaluationTypeTest"], *args_val)
                erro_v = best_model.score(yd, yp, parametros_mggp["evaluationMode"])
            except Exception:
                pass
            
            if 0.0 < erro_v < 100.0:
                nome_modelo = f"modelo_rmse_{modelos_aprovados + 1}.pkl" 
                with open(os.path.join(pasta_training, nome_modelo), 'wb') as f:
                    dill.dump(best_model, f)
                
                # [NOVIDADE]: Registra a aprovação na planilha local e salva!
                etapa_atual = f"Treino_{modelos_aprovados + 1:02d}"
                df_local.loc[df_local['Etapa'] == etapa_atual, 'Status'] = 'Concluído'
                df_local.to_excel(caminho_check_local, index=False)

                modelos_aprovados += 1
                print(f" -> [Treino {t_id}] Modelo {modelos_aprovados} APROVADO! RMSE: {erro_v:.4f}")
            else:
                print(f" -> [Treino {t_id}] Modelo REPROVADO! RMSE: {erro_v:.4f}")
                
            if os.path.exists(caminho_temp): os.remove(caminho_temp)
            del mggp
            gc.collect()
            
        # --- 4. VALIDA OS 10 TREINAMENTOS COM CHECKPOINT INDIVIDUAL ---
        print(f" -> [Treino {t_id}] Iniciando validação em massa (Lendo Checkpoint)...")
        
        for i in range(1, 11):
            etapa_val_atual = f"Validacao_{i:02d}"
            status_val = df_local.loc[df_local['Etapa'] == etapa_val_atual, 'Status'].values[0]
            
            # [NOVIDADE]: Se já validou antes da energia cair, pula!
            if status_val == 'Concluído':
                print(f" -> [Treino {t_id}] {etapa_val_atual} já estava concluída. Pulando...")
                continue
                
            nome_arquivo = f"modelo_rmse_{i}.pkl"
            caminho_modelo = os.path.join(pasta_training, nome_arquivo)
            
            if not os.path.exists(caminho_modelo):
                continue # Se o modelo não existe por algum motivo, previne erro.
            
            try:
                with open(caminho_modelo, 'rb') as f:
                    modelo_real = dill.load(f)
            except Exception: 
                continue
                
            fig, axes = plt.subplots(4, 2, figsize=(18, 20))
            fig.suptitle(f"Validação Completa (Dados Originais): {nome_arquivo}", fontsize=16, y=1.01)
            resultados_csv_parcial = []
            
            for idx_pista, (nome_pista, caminho_pista) in enumerate(pistas_validacao_massa.items()):
                u_val_raw, y_val_raw = carregar_dados_bicicleta(caminho_pista)
                try:
                    args_val_massa = (y_val_raw, u_val_raw) if parametros_mggp["evaluationTypeTest"] != "MShooting" else (parametros_mggp["k"], y_val_raw, u_val_raw)
                    yp_raw, yd_raw = modelo_real.predict(parametros_mggp["evaluationTypeTest"], *args_val_massa)
                    
                    min_len = min(len(yd_raw), len(yp_raw))
                    yd_raw, yp_raw = yd_raw[:min_len], yp_raw[:min_len]
                    
                    rmse_vx = calcular_rmse(yd_raw[:, 0], yp_raw[:, 0])
                    rmse_vy = calcular_rmse(yd_raw[:, 1], yp_raw[:, 1])
                    rmse_medio = (rmse_vx + rmse_vy) / 2
                except Exception:
                    yp_raw, yd_raw = np.zeros_like(y_val_raw), y_val_raw
                    rmse_vx, rmse_vy, rmse_medio = 999.9, 999.9, 999.9
                    
                resultados_csv_parcial.append({
                    "Modelo": nome_arquivo, "Pista": nome_pista,
                    "RMSE_Vx": round(rmse_vx, 4), "RMSE_Vy": round(rmse_vy, 4), "RMSE_Médio": round(rmse_medio, 4)
                })
                
                axes[idx_pista, 0].plot(yd_raw[:, 0], color='black', label="Real")
                axes[idx_pista, 0].plot(yp_raw[:, 0], color='blue', linestyle='--', label="Previsto")
                axes[idx_pista, 0].set_title(f"{nome_pista} - Vx | RMSE: {rmse_vx:.4f}")
                axes[idx_pista, 0].grid(True, linestyle=':', alpha=0.6)
                
                axes[idx_pista, 1].plot(yd_raw[:, 1], color='black', label="Real")
                axes[idx_pista, 1].plot(yp_raw[:, 1], color='red', linestyle='--', label="Previsto")
                axes[idx_pista, 1].set_title(f"{nome_pista} - Vy | RMSE: {rmse_vy:.4f}")
                axes[idx_pista, 1].grid(True, linestyle=':', alpha=0.6)
                
            plt.tight_layout()
            plt.savefig(os.path.join(pasta_salvamento, nome_arquivo.replace(".pkl", ".png")), bbox_inches='tight')
            plt.close(fig)
            
            # [NOVIDADE]: Salva os resultados desta validação anexando ao CSV existente
            if len(resultados_csv_parcial) > 0:
                df_novo = pd.DataFrame(resultados_csv_parcial)
                caminho_csv = os.path.join(pasta_salvamento, "relatorio_validacao_rmse.csv")
                if os.path.exists(caminho_csv):
                    df_existente = pd.read_csv(caminho_csv, sep=';', decimal=',')
                    df_combinado = pd.concat([df_existente, df_novo], ignore_index=True)
                    df_combinado.to_csv(caminho_csv, index=False, sep=';', decimal=',')
                else:
                    df_novo.to_csv(caminho_csv, index=False, sep=';', decimal=',')

            # [NOVIDADE]: Marca a validação atual como concluída na planilha local e salva!
            df_local.loc[df_local['Etapa'] == etapa_val_atual, 'Status'] = 'Concluído'
            df_local.to_excel(caminho_check_local, index=False)
            print(f" -> [Treino {t_id}] {etapa_val_atual} finalizada e salva.")

        # --- 5. ATUALIZA A PLANILHA GLOBAL (O Maestro) ---
        df_controle = pd.read_excel(CAMINHO_PLANILHA)
        df_controle.loc[df_controle['ID'] == t_id, 'Status'] = 'Concluído'
        df_controle.to_excel(CAMINHO_PLANILHA, index=False)
        print(f" -> [Planilha Global] ID {t_id} finalizado 100%! Avançando...")

        # --- 6. LIMPA O CACHE E A RAM ---
        plt.close('all')
        try:
            del u_train, y_train, u_val, y_val
        except NameError:
            pass
        gc.collect()
        print(f" -> [🧹 RAM] Faxina de cache concluída para o ID {t_id}.\n")










 📑  SISTEMA AUTÔNOMO MGGP — LENDO PLANILHA EM TEMPO REAL
 📈 Restam: 29 treinamentos pendentes na fila.
 🚀 Processando: [ ID: 21 | nTerms: 3 | maxHeight: 7 ]
 🎯 Fábrica: [ Aprovados: 0/10 | Tentativa Atual: 1/50 ]
 📜 LOGS DA RODADA ATUAL:
-------------------------------------------------------------------------------------
System Mode: MIMO. N° Inputs: 5. N° Outputs: 2
Input Samples: 42001. Output Samples: 42001



Evaluating Initial Population: 100%|██████████| 300/300 [00:11<00:00, 26.38it/s]


   	     	          fitness          
   	     	---------------------------
gen	evals	min     	avg	max
1  	300  	0.325313	inf	inf


Evaluating Population: 100%|██████████| 232/232 [00:08<00:00, 26.16it/s]


2  	232  	0.325313	inf	inf


...

Evaluating Population: 100%|██████████| 222/222 [00:08<00:00, 25.55it/s]


9  	222  	0.21705 	inf        	inf        


Evaluating Population: 100%|██████████| 223/223 [00:08<00:00, 24.80it/s]


10 	223  	0.21705 	inf        	inf        


Evaluating Population:  84%|████████▍ | 201/238 [00:07<00:01, 26.34it/s]